In [ ]:
import xarray as xr
import numpy as np
import glob

In [2]:
TRAIN_DIR = '../../climatenet/train/'
TEST_DIR  = '../../climatenet/test/'
 
SHAPE = (768, 1152)

In [3]:
# Chargement des matrices déja calculés dans climatenet_exploration.ipynb
count_tc_train = np.load('../npy_files/count_tc.npy')
count_ar_train = np.load('../npy_files/count_ar.npy')
n_train = len(sorted(glob.glob(f'{TRAIN_DIR}*.nc')))

In [6]:
# count sur les fichiers de test
files_test = sorted(glob.glob(f'{TEST_DIR}*.nc'))
n_test = len(files_test)
print(f'\nNombre de fichiers test trouvés : {n_test}')
 
count_tc_test = np.zeros(SHAPE, dtype=np.int32)
count_ar_test = np.zeros(SHAPE, dtype=np.int32)
 
for f in files_test:
    ds = xr.open_dataset(f)
    labels = ds['LABELS'].values
    ds.close()
    
    count_tc_test += (labels == 1).astype(np.int32)
    count_ar_test += (labels == 2).astype(np.int32)
 
np.save('../npy_files/count_tc_test.npy', count_tc_test)
np.save('../npy_files/count_ar_test.npy', count_ar_test)


Nombre de fichiers test trouvés : 61


In [7]:
# Statistique de test
total_pixels = SHAPE[0] * SHAPE[1]
 
never_tc_test  = count_tc_test == 0
never_ar_test  = count_ar_test == 0
never_event_test = never_tc_test & never_ar_test
 
print(f'Total pixels par grille: {total_pixels}')
print(f'Jamais TC  (sur {n_test} fichiers): {never_tc_test.sum()}  ({100*never_tc_test.mean():.1f}%)')
print(f'Jamais AR  (sur {n_test} fichiers): {never_ar_test.sum()}  ({100*never_ar_test.mean():.1f}%)')
print(f'Jamais TC ni AR: {never_event_test.sum()}  ({100*never_event_test.mean():.1f}%)')

Total pixels par grille: 884736
Jamais TC  (sur 61 fichiers): 786212  (88.9%)
Jamais AR  (sur 61 fichiers): 477240  (53.9%)
Jamais TC ni AR: 434593  (49.1%)


In [10]:
# Comparaison entre train et test
never_event_train = (count_tc_train == 0) & (count_ar_train == 0)
 
safe_mask = never_event_train & never_event_test

print(f'Zones mortes dans TRAIN: {never_event_train.sum()}  ({100*never_event_train.mean():.1f}%)')
print(f'Zones mortes dans TEST: {never_event_test.sum()}  ({100*never_event_test.mean():.1f}%)')
print()
print(f'Zones mortes dans les DEUX   : {safe_mask.sum()}  ({100*safe_mask.mean():.1f}%): safe_mask.npy')
print()
 
 
# Sauvegarde du safe_mask
np.save('../npy_files/safe_mask.npy', safe_mask)
print(f'Masque sauvegardé dans safe_mask.npy')

Zones mortes dans TRAIN: 247154  (27.9%)
Zones mortes dans TEST: 434593  (49.1%)

Zones mortes dans les DEUX   : 238991  (27.0%): safe_mask.npy

Masque sauvegardé dans safe_mask.npy


In [9]:
safe_mask.shape

(768, 1152)